In [0]:
# Standard calendar (date dimension) table: 1940-01-01 to 2030-12-31
target_table = "adb_neeleshcaresync.silver.calendar"
start_date, end_date = "1940-01-01", "2030-12-31"

spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Standard calendar / date dimension from {start_date} to {end_date}'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'{start_date}', DATE'{end_date}', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)              AS date_key,
  d                                                    AS full_date,
  -- Day
  dayofmonth(d)                                        AS day_of_month,
  extract(DAYOFWEEK_ISO FROM d)                        AS day_of_week,        -- 1=Mon ... 7=Sun
  date_format(d, 'EEEE')                               AS day_name,
  date_format(d, 'EEE')                                AS day_name_short,
  dayofyear(d)                                         AS day_of_year,
  extract(DAYOFWEEK_ISO FROM d) IN (6, 7)              AS is_weekend,
  extract(DAYOFWEEK_ISO FROM d) NOT IN (6, 7)          AS is_weekday,
  -- Week (ISO 8601, Monday start)
  weekofyear(d)                                        AS week_of_year,
  extract(YEAROFWEEK FROM d)                           AS iso_year,
  CAST(date_trunc('WEEK', d) AS DATE)                  AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)     AS week_end_date,
  CAST(ceil(dayofmonth(d) / 7.0) AS INT)               AS week_of_month,
  -- Month
  month(d)                                             AS month,
  date_format(d, 'MMMM')                               AS month_name,
  date_format(d, 'MMM')                                AS month_name_short,
  date_format(d, 'yyyy-MM')                            AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                AS year_month_key,
  trunc(d, 'MM')                                       AS month_start_date,
  last_day(d)                                          AS month_end_date,
  dayofmonth(last_day(d))                              AS days_in_month,
  d = trunc(d, 'MM')                                   AS is_month_start,
  d = last_day(d)                                      AS is_month_end,
  -- Quarter
  quarter(d)                                           AS quarter,
  concat('Q', quarter(d))                              AS quarter_name,
  concat(year(d), '-Q', quarter(d))                    AS year_quarter,
  CAST(date_trunc('QUARTER', d) AS DATE)               AS quarter_start_date,
  date_add(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), -1) AS quarter_end_date,
  datediff(d, CAST(date_trunc('QUARTER', d) AS DATE)) + 1 AS day_of_quarter,
  -- Half year
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END            AS half_year,
  concat(year(d), '-H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END) AS year_half,
  -- Year
  year(d)                                              AS year,
  trunc(d, 'YEAR')                                     AS year_start_date,
  make_date(year(d), 12, 31)                           AS year_end_date,
  (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 AS is_leap_year,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 THEN 366 ELSE 365 END AS days_in_year,
  floor(year(d) / 10) * 10                             AS decade
FROM dates
""")

print(f"Created {target_table}")

In [0]:
# Validate the calendar table against expected values
from datetime import date

cal = spark.table(target_table)
expected_days = (date.fromisoformat(end_date) - date.fromisoformat(start_date)).days + 1

stats = spark.sql(f"""
SELECT
  min(full_date)                                   AS min_date,
  max(full_date)                                   AS max_date,
  count(*)                                         AS row_count,
  count(DISTINCT date_key)                         AS distinct_date_keys,
  count_if(date_key IS NULL OR full_date IS NULL)  AS null_keys,
  count_if(month = 2 AND day_of_month = 29)        AS leap_days,
  count_if(is_leap_year) / 366                     AS leap_years,
  count_if(is_weekend = is_weekday)                AS weekend_flag_conflicts
FROM {target_table}
""").first()

checks = {
    "Start date is 1940-01-01": str(stats.min_date) == start_date,
    "End date is 2030-12-31": str(stats.max_date) == end_date,
    f"Row count = {expected_days} (one row per day)": stats.row_count == expected_days,
    "date_key is unique": stats.distinct_date_keys == stats.row_count,
    "No null keys": stats.null_keys == 0,
    "Leap days match leap years": stats.leap_days == stats.leap_years,
    "Weekend/weekday flags consistent": stats.weekend_flag_conflicts == 0,
}

for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(stats.asDict())
assert all(checks.values()), "Calendar validation failed"

In [0]:
# Preview calendar rows
display(spark.table(target_table).orderBy("date_key").limit(100))